# ESG Step 5 CV Colab 執行筆記本

用途：在 Colab 上執行 Step 5 LoRA fine-tuning 與 validation inference。實驗設定仍由 `config/config.yaml` 控制；本 notebook 只在開頭集中設定重要參數，然後自動寫入 Colab runtime 的 `config.yaml`。

若使用 `ESG_step5_cv_colab_bundle.zip`，請先在 Colab 解壓到 `/content`，使專案根目錄成為 `/content/ESG-sft`。

執行前請確認專案資料已放在 `PROJECT_ROOT`，至少需要：

- `src/step4_cv/`
- `src/step5_cv/`
- `src/step5_reasoning/`
- `data/processed/step5_sft/{annotator_name}/manifest.json`
- `data/processed/step5_sft/{annotator_name}/fold_*/train_sft_text.json`
- `data/processed/step5_sft/{annotator_name}/fold_*/val_eval.json`


In [ ]:
from pathlib import Path

# 路徑設定
PROJECT_ROOT = Path('/content/ESG-sft')
USE_GOOGLE_DRIVE = True
DRIVE_ROOT = Path('/content/drive/MyDrive/ESG-sft')

# 執行控制
INSTALL_DEPENDENCIES = True
RUN_TRAINING = True
RUN_INFERENCE = True

# 模型：gemma | llama | qwen | ministral
MODEL_TYPE = 'llama'

# 實驗：balance_train_only | no_balance | combined
# balance_train_only: blance_500 僅加入 train，對應 p2_balance
# no_balance: 不加入 blance_500，對應 p2
# combined: blance_500 合併後再 split，對應 p2_combined
EXPERIMENT = 'balance_train_only'
ANNOTATOR = 'p2'

# LoRA / training
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 5
NUM_TRAIN_EPOCHS = 15
LEARNING_RATE = 2e-4
OPTIM = 'adamw_8bit'
WEIGHT_DECAY = 0.01
LR_SCHEDULER_TYPE = 'linear'
RANDOM_STATE = 3407
MAX_SEQ_LENGTH = 2048

# Runtime / checkpoint
DTYPE = 'auto'
USE_GRADIENT_CHECKPOINTING = 'auto'
VISION_ATTN_IMPLEMENTATION = 'auto'
CHECKPOINT_SAVE_STRATEGY = 'auto'
SAVE_TOTAL_LIMIT = None
SAVE_STEPS = None
SAVE_FINAL_CHECKPOINT_COPY = False
MINIMUM_FREE_SPACE_GB = 2
RESUME = True

# Ablation
HUMAN_ONLY = False
LABEL_ONLY = False
SYNTHETIC_ONLY = False
VARIANT_NAME = 'default'
NON_ESG_CAP = 250

# X/Y/Z plot
XYZ_ENABLED = False
XYZ_EPOCHS = [15]

# Inference
CHECKPOINT_STRATEGY = 'final'  # final | latest | epoch | best_epoch
CHECKPOINT_EPOCH = None
BEST_EPOCH_METRIC = 'macro_f1'
INFERENCE_VRAM_FRACTION = None
MAX_NEW_TOKENS = 512
MAX_NEW_TOKENS_LABEL_ONLY = 16

# Dry run：先用少量資料確認 Colab 環境、路徑、adapter 輸出都正常
DRY_RUN_ENABLED = False
DRY_RUN_FOLD = 0
DRY_RUN_TRAIN_SAMPLES = 8
DRY_RUN_VAL_SAMPLES = 4
DRY_RUN_MAX_STEPS = 1
DRY_RUN_RESULTS_SUBDIR = 'dry_run'

# Hugging Face token 可在 Colab secrets 或環境變數設定；必要時也可直接填字串。
HF_TOKEN = ''


In [ ]:
import os
import sys

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {exc}')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

os.environ.setdefault('WANDB_DISABLED', 'true')
os.environ.setdefault('UNSLOTH_DISABLE_STATISTICS', '1')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'PROJECT_ROOT 不存在：{PROJECT_ROOT}。請先上傳或解壓專案到此路徑。')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DRIVE_ROOT = {DRIVE_ROOT}')


In [ ]:
if INSTALL_DEPENDENCIES:
    import subprocess
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'uv'], check=True)
    subprocess.run(['uv', 'pip', 'install', '--system', '-e', str(PROJECT_ROOT)], check=True)
else:
    print('略過依賴安裝。')


In [ ]:
import yaml

EXPERIMENT_SETTINGS = {
    'balance_train_only': {'balance_enabled': True, 'include_in_cv': False},
    'no_balance': {'balance_enabled': False, 'include_in_cv': False},
    'combined': {'balance_enabled': True, 'include_in_cv': True},
}
if EXPERIMENT not in EXPERIMENT_SETTINGS:
    raise ValueError(f'Unsupported EXPERIMENT: {EXPERIMENT}')

exp_cfg = EXPERIMENT_SETTINGS[EXPERIMENT]
models_root = DRIVE_ROOT / 'models' / 'step5_cv' if USE_GOOGLE_DRIVE else PROJECT_ROOT / 'models' / 'step5_cv'
results_root = DRIVE_ROOT / 'results' / 'step5_cv' if USE_GOOGLE_DRIVE else PROJECT_ROOT / 'results' / 'step5_cv'

config_payload = {
    'step4_cv': {
        'annotator': ANNOTATOR,
        'annotators': {
            'p1': {'csv': 'data/origin_data/10k_1A/project-16-at-2026-03-31-11-52-a8826fd3.csv'},
            'p2': {'csv': 'data/origin_data/10k_1A/chiang_500.csv'},
        },
        'balance': {
            'enabled': exp_cfg['balance_enabled'],
            'include_in_cv': exp_cfg['include_in_cv'],
            'csv': 'data/origin_data/10k_1A/blance_500.csv',
        },
        'paths': {
            'classified_json': 'data/processed/step2_classification/classified.json',
            'output_dir': 'data/processed/step4_cv',
            'folds_path': 'data/processed/step4_cv/cv_folds.json',
        },
        'folds': {'n_splits': 3, 'shuffle': True, 'random_state': 42},
        'thresholds': {'target_accuracy': 0.95, 'min_samples_global': 30, 'min_samples_per_class': 8},
        'pseudo_labels': {'enabled': False, 'total_target': 2500, 'allocation_alpha': 0.5, 'caps': {'Non-ESG': 1000}},
        'balancing': {'alpha': 0.5, 'total_budget': 4000},
        'synthetic_generation': {
            'enabled': False,
            'seed_examples_per_class': 3,
            'boundary_focus_labels': ['Product Liability', 'Climate Change', 'Corporate Governance'],
            'output_filename': 'synthetic_data.json',
            'provider': 'openai',
            'model': 'gpt-5.4-mini-2026-03-17',
            'api_key_env': 'OPENAI_API_KEY',
            'max_samples_per_call': 20,
            'max_retries': 5,
            'reasoning_effort': 'low',
            'sleep_seconds': 0.05,
            'retry_backoff_seconds': 0.5,
            'refresh_after_generation': True,
            'report_filename': 'synthetic_generation_report.json',
            'max_output_tokens': 16000,
        },
    },
    'step5_reasoning': {
        'paths': {
            'step4_output_dir': 'data/processed/step4_cv',
            'output_dir': 'data/processed/step5_reasoning',
            'sft_output_dir': 'data/processed/step5_sft',
        },
        'generation': {'enabled': False, 'run_reasoning_generation': False},
        'sft': {'run_sft_build': False, 'shuffle_seed': 3407},
    },
    'step5_cv': {
        'finbert': {'step4_output_dir': 'data/processed/step4_cv', 'results_dir': str(results_root / 'finbert')},
        'api_llm': {'enabled': False, 'step4_output_dir': 'data/processed/step4_cv', 'results_root': str(results_root), 'models': {}},
        'finetune': {
            'sft_output_dir': 'data/processed/step5_sft',
            'results_root': str(results_root),
            'models_root': str(models_root),
            'resume': RESUME,
            'model_registry': {
                'gemma': 'unsloth/gemma-4-E4B-it',
                'llama': 'unsloth/Llama-3.2-3B-Instruct',
                'qwen': 'unsloth/Qwen3.5-4B',
                'ministral': 'unsloth/Ministral-3-3B-Instruct-2512',
            },
            'model_overrides': {},
            'training': {
                'lora_r': LORA_R,
                'lora_alpha': LORA_ALPHA,
                'lora_dropout': LORA_DROPOUT,
                'per_device_train_batch_size': PER_DEVICE_TRAIN_BATCH_SIZE,
                'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
                'warmup_steps': WARMUP_STEPS,
                'num_train_epochs': NUM_TRAIN_EPOCHS,
                'learning_rate': LEARNING_RATE,
                'optim': OPTIM,
                'weight_decay': WEIGHT_DECAY,
                'lr_scheduler_type': LR_SCHEDULER_TYPE,
                'random_state': RANDOM_STATE,
                'max_seq_length': MAX_SEQ_LENGTH,
            },
            'runtime': {
                'dtype': DTYPE,
                'use_gradient_checkpointing': USE_GRADIENT_CHECKPOINTING,
                'vision_attn_implementation': VISION_ATTN_IMPLEMENTATION,
            },
            'checkpointing': {
                'save_strategy': CHECKPOINT_SAVE_STRATEGY,
                'save_total_limit': SAVE_TOTAL_LIMIT,
                'save_steps': SAVE_STEPS,
                'save_final_checkpoint_copy': SAVE_FINAL_CHECKPOINT_COPY,
                'minimum_free_space_gb': MINIMUM_FREE_SPACE_GB,
            },
            'ablation': {
                'human_only': HUMAN_ONLY,
                'label_only': LABEL_ONLY,
                'synthetic_only': SYNTHETIC_ONLY,
                'variant_name': VARIANT_NAME,
                'non_esg_cap': NON_ESG_CAP,
            },
            'xyz_plot': {'enabled': XYZ_ENABLED, 'epochs': XYZ_EPOCHS},
            'inference': {
                'checkpoint_strategy': CHECKPOINT_STRATEGY,
                'checkpoint_epoch': CHECKPOINT_EPOCH,
                'best_epoch_metric': BEST_EPOCH_METRIC,
                'vram_fraction': INFERENCE_VRAM_FRACTION,
                'max_new_tokens': MAX_NEW_TOKENS,
                'max_new_tokens_label_only': MAX_NEW_TOKENS_LABEL_ONLY,
            },
            'dry_run': {
                'enabled': DRY_RUN_ENABLED,
                'fold': DRY_RUN_FOLD,
                'train_samples': DRY_RUN_TRAIN_SAMPLES,
                'val_samples': DRY_RUN_VAL_SAMPLES,
                'max_steps': DRY_RUN_MAX_STEPS,
                'results_subdir': DRY_RUN_RESULTS_SUBDIR,
            },
        },
        'evaluation': {'results_root': str(results_root)},
    },
}

config_path = PROJECT_ROOT / 'config' / 'config.yaml'
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(config_payload, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(f'已寫入 Colab runtime config: {config_path}')
print(f'models_root = {models_root}')
print(f'results_root = {results_root}')


In [ ]:
import json

from src.step4_cv.common import load_step4_config, resolve_annotator_name

annotator_name = resolve_annotator_name(load_step4_config())
manifest_path = PROJECT_ROOT / 'data' / 'processed' / 'step5_sft' / annotator_name / 'manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(
        f'找不到 SFT manifest: {manifest_path}\n'
        f'請確認已打包 data/processed/step5_sft/{annotator_name}/ 到 PROJECT_ROOT。'
    )

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(f'annotator_name = {annotator_name}')
print(f'manifest = {manifest_path}')
for fold in manifest['folds']:
    sft_path = PROJECT_ROOT / fold['sft_path']
    val_path = PROJECT_ROOT / fold['val_eval_path']
    print(f"fold_{fold['fold']}: sft_size={fold['sft_size']} sft_exists={sft_path.exists()} val_exists={val_path.exists()}")


In [ ]:
if RUN_TRAINING:
    from src.step5_cv.run_cv_finetune import run_finetune
    run_finetune(MODEL_TYPE)
else:
    print('略過 training。')

if RUN_INFERENCE:
    from src.step5_cv.run_cv_inference import run_inference_only
    run_inference_only(MODEL_TYPE)
else:
    print('略過 inference。')


In [ ]:
from IPython.display import Markdown, display

from src.step5_cv.common import load_step5_cv_config
from src.step5_cv.run_cv_finetune import (
    apply_model_overrides,
    get_ablation_cfg,
    get_inference_cfg,
    resolve_inference_run_suffix,
    resolve_variant_name,
)

cfg = apply_model_overrides(load_step5_cv_config()['finetune'], MODEL_TYPE)
ablation_cfg = get_ablation_cfg(cfg)
variant_name = resolve_variant_name(ablation_cfg)
inference_suffix = resolve_inference_run_suffix(get_inference_cfg(cfg))

result_dir = Path(cfg['results_root']) / annotator_name / MODEL_TYPE
if variant_name != 'default':
    result_dir = result_dir / variant_name
if inference_suffix:
    result_dir = result_dir / inference_suffix
if DRY_RUN_ENABLED:
    result_dir = result_dir / DRY_RUN_RESULTS_SUBDIR

summary_md = result_dir / 'overall_summary.md'
print(f'result_dir = {result_dir}')
if summary_md.exists():
    display(Markdown(summary_md.read_text(encoding='utf-8')))
else:
    print('目前沒有 overall_summary.md。dry_run 或只跑 training 時這是正常的。')


In [ ]:
import shutil

archive_base = DRIVE_ROOT / 'archives' / f'step5_cv_{annotator_name}_{MODEL_TYPE}' if USE_GOOGLE_DRIVE else PROJECT_ROOT / 'archives' / f'step5_cv_{annotator_name}_{MODEL_TYPE}'
archive_base.parent.mkdir(parents=True, exist_ok=True)
if result_dir.exists():
    archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=result_dir)
    print(f'已打包結果：{archive_path}')
else:
    print(f'結果目錄不存在，略過打包：{result_dir}')
